In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, classification_report,
    confusion_matrix, precision_score, recall_score,
    f1_score, average_precision_score, precision_recall_curve
)
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Case Study 2: Credit Card Fraud Detection

Apply XGBoost on a heavily imbalanced transactions dataset. Use SMOTE for oversampling, tune the decision threshold, and interpret the results using feature importance scores.

In [ ]:
# Install the required machine learning libraries
!pip install -q xgboost imbalanced-learn

First, let's upload the credit card transactions dataset and load it into a Pandas DataFrame.

In [ ]:
from google.colab import files

# Upload creditcard.csv
uploaded = files.upload()

# Load the CSV file
df = pd.read_csv("creditcard.csv")

print("Dataset Shape:", df.shape)
df.head()

Let's explore the dataset and check the distribution of genuine and fraudulent transactions.

In [ ]:
print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())

print("\nMissing Values:")
print(df.isnull().sum().sum())

print("\nTransaction Class Counts:")
print(df["Class"].value_counts())

print("\nTransaction Class Percentage:")
print(df["Class"].value_counts(normalize=True) * 100)

# Visualizing the class distribution
plt.figure(figsize=(7, 5))
df["Class"].value_counts().plot(kind="bar")
plt.title("Genuine vs Fraudulent Transactions")
plt.xlabel("Class (0 = Genuine, 1 = Fraud)")
plt.ylabel("Number of Transactions")
plt.xticks(rotation=0)
plt.grid(axis="y")
plt.show()

The dataset is highly imbalanced because fraudulent transactions are much fewer than genuine transactions. We split the data first and apply SMOTE only to the training set.

In [ ]:
# Separate features and target variable
X = df.drop("Class", axis=1)
y = df["Class"]

# Split into training, validation, and testing data
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

print("Training Set:", X_train.shape)
print("Validation Set:", X_val.shape)
print("Testing Set:", X_test.shape)

Before applying SMOTE, we standardize the features. SMOTE works by creating synthetic samples of the minority class.

In [ ]:
# Standardize the feature values
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

X_train_scaled = pd.DataFrame(
    X_train_scaled, columns=X_train.columns, index=X_train.index
)
X_val_scaled = pd.DataFrame(
    X_val_scaled, columns=X_val.columns, index=X_val.index
)
X_test_scaled = pd.DataFrame(
    X_test_scaled, columns=X_test.columns, index=X_test.index
)

print("Class distribution before SMOTE:")
print(y_train.value_counts())

# Apply SMOTE only to the training data
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print("\nClass distribution after SMOTE:")
print(y_train_smote.value_counts())

Now let's train an XGBoost classifier on the SMOTE-balanced training data.

In [ ]:
model_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(X_train_smote, y_train_smote)

print("XGBoost model trained successfully!")

Finally, let's evaluate the XGBoost model using the default decision threshold of 0.50.

In [ ]:
# Predict fraud probabilities
y_val_proba = model_xgb.predict_proba(X_val_scaled)[:, 1]
y_test_proba = model_xgb.predict_proba(X_test_scaled)[:, 1]

# Default threshold = 0.50
y_pred_xgb = (y_test_proba >= 0.50).astype(int)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))

print("\nROC AUC Score:", roc_auc_score(y_test, y_test_proba))
print("PR AUC Score:", average_precision_score(y_test, y_test_proba))

Because fraud detection is highly imbalanced, the 0.50 threshold may not be the best choice. Let's test different thresholds using the validation set and select the threshold with the best F1-score.

In [ ]:
thresholds = np.arange(0.05, 0.96, 0.05)
threshold_results = []

for threshold in thresholds:
    y_val_pred = (y_val_proba >= threshold).astype(int)

    precision = precision_score(y_val, y_val_pred, zero_division=0)
    recall = recall_score(y_val, y_val_pred, zero_division=0)
    f1 = f1_score(y_val, y_val_pred, zero_division=0)

    threshold_results.append([threshold, precision, recall, f1])

threshold_df = pd.DataFrame(
    threshold_results,
    columns=["Threshold", "Precision", "Recall", "F1 Score"]
)

print(threshold_df)

best_row = threshold_df.loc[threshold_df["F1 Score"].idxmax()]
best_threshold = float(best_row["Threshold"])

print("\nBest Decision Threshold:", best_threshold)
print("Best Validation Precision:", best_row["Precision"])
print("Best Validation Recall:", best_row["Recall"])
print("Best Validation F1 Score:", best_row["F1 Score"])

In [ ]:
# Plot Precision, Recall and F1-score for different thresholds
plt.figure(figsize=(8, 6))
plt.plot(threshold_df["Threshold"], threshold_df["Precision"], marker="o", label="Precision")
plt.plot(threshold_df["Threshold"], threshold_df["Recall"], marker="o", label="Recall")
plt.plot(threshold_df["Threshold"], threshold_df["F1 Score"], marker="o", label="F1 Score")
plt.xlabel("Decision Threshold")
plt.ylabel("Score")
plt.title("Threshold Tuning")
plt.legend()
plt.grid(True)
plt.show()

Now we use the selected threshold on the unseen test data and calculate the final performance.

In [ ]:
# Apply the best threshold to the test probabilities
y_pred_final = (y_test_proba >= best_threshold).astype(int)

accuracy = (y_pred_final == y_test).mean()
precision = precision_score(y_test, y_pred_final, zero_division=0)
recall = recall_score(y_test, y_pred_final, zero_division=0)
f1 = f1_score(y_test, y_pred_final, zero_division=0)
roc_auc = roc_auc_score(y_test, y_test_proba)
pr_auc = average_precision_score(y_test, y_test_proba)

print("\n========== FINAL RESULTS ==========")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", roc_auc)
print("PR-AUC   :", pr_auc)
print("Threshold:", best_threshold)

print("\nFinal Classification Report:")
print(classification_report(y_test, y_pred_final))

print("\nFinal Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

In [ ]:
# Plot the final confusion matrix
cm = confusion_matrix(y_test, y_pred_final)

plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("Final Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ["Genuine", "Fraud"])
plt.yticks([0, 1], ["Genuine", "Fraud"])
plt.xlabel("Predicted")
plt.ylabel("Actual")

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()

Let's visualize the ROC curve and Precision-Recall curve. PR-AUC is particularly useful for highly imbalanced fraud datasets.

In [ ]:
# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

# Precision-Recall Curve
precision_curve, recall_curve, _ = precision_recall_curve(y_test, y_test_proba)

plt.figure(figsize=(8, 6))
plt.plot(recall_curve, precision_curve, label=f"PR Curve (AUC = {pr_auc:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend(loc="lower left")
plt.grid(True)
plt.show()

Now let's interpret the XGBoost model using feature importance scores. Higher importance means the feature contributed more to the model's decisions.

In [ ]:
# Calculate feature importance scores
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": model_xgb.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance", ascending=False
)

print("Top 15 Important Features:")
print(feature_importance.head(15))

# Plot top 15 features
top_features = feature_importance.head(15).sort_values("Importance")

plt.figure(figsize=(10, 7))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Top 15 Feature Importance Scores - XGBoost")
plt.grid(axis="x")
plt.show()

### Conclusion

XGBoost was applied to detect fraudulent credit card transactions in a highly imbalanced dataset. SMOTE was used on the training data to improve learning of the minority fraud class. Different decision thresholds were tested on the validation data, and the best threshold was selected using F1-score. The final model was evaluated using Precision, Recall, F1-score, ROC-AUC, PR-AUC, and a confusion matrix. Feature importance scores were also used to identify the features that contributed most to fraud prediction.

In fraud detection, recall is important because a false negative means that an actual fraudulent transaction was missed. Precision is also important because too many false positives can incorrectly flag genuine customers.